<h1>Livrable 4.1 : Analyse prédictive de données non-structurées par l'intelligence artificielle</h1>

> **Livrable attendu :** Un code source incluant la conception de l'algorithme et les métriques de performances sur des données de validation.

# CQT MLP Trainer

L'exécution de ce notebook a pour prérequis :

- le téléchargement des dépendances :
  
  ```bash
  uv sync --group audio_midi --group api
  ```

- le téléchargement du dataset `GuitarSet` :
  
  ```bash
  uv run ./audio_midi/main.py --download_datasets --no_idmt_smt_guitar
  ```

- le démarrage de l'infrastructure docker :
  
  ```bash
  cp .env.exemple .env
  docker-compose up -d
  ```
  
- l'ingestion du dataset `GuitarSet` :
  
  ```bash
  uv run ./audio_midi/main.py --ingest_guitar_set
  ```
  
- le prétraitement du dataste `GuitarSet` :
  
  ```bash
  uv run ./audio_midi/main.py --preprocess_datasets --no_idmt_smt_guitar
  ```


## Imports

In [1]:
import sys
from pathlib import Path

APP_DIR = Path.cwd().parent
sys.path.append(APP_DIR.as_posix())

In [2]:
# Chemins
OUTPUT_DIR = APP_DIR / "output"
ARTIFACT_DIR = OUTPUT_DIR / "cqt_mlp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Imports graphiques
import matplotlib.pyplot as plt
import seaborn as sns

# Accessibilité : Daltonisme, Dyslexie, Confort Visuel
sns.set_theme(
    style="whitegrid",
    palette="colorblind",
    context="notebook",
)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.family": "Arial",
        "font.size": 12,
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.labelsize": 13,
        "axes.labelweight": "medium",
        "axes.edgecolor": "black",
        "axes.linewidth": 1.2,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "lines.linewidth": 2.2,
        "lines.markersize": 7,
        "legend.fontsize": 11,
        "legend.frameon": True,
        "legend.framealpha": 0.95,
        "grid.linestyle": ":",
        "grid.linewidth": 0.8,
        "grid.alpha": 0.6,
    }
)

COLORBLIND_PALETTE = sns.color_palette("colorblind")

In [4]:
import os
import json
import warnings
import logging
from datetime import datetime
from time import perf_counter
import functools

import numpy as np
import pandas as pd

import tensorflow as tf
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_curve,
    hamming_loss,
    accuracy_score,
)
from sklearn.preprocessing import RobustScaler

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.entities import Experiment
from mlflow.models import infer_signature

from src.pipelines import DatasetBuilderPipeline
from settings.dataset_builder_pipeline_settings import DatasetBuilderPipelineSettings
from settings import MLFLOW_SETTINGS, GUITAR_SET_SETTINGS

warnings.filterwarnings("ignore")

RANDOM_STATE = 73
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

c:\Users\Administrateur\Documents\M2i_CDSD_Projet\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\Administrateur\Documents\M2i_CDSD_Projet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
os.environ["AWS_ACCESS_KEY_ID"] = MLFLOW_SETTINGS.aws_access_key_id
os.environ["AWS_SECRET_ACCESS_KEY"] = MLFLOW_SETTINGS.aws_secret_access_key
os.environ["MLFLOW_S3_ENDPOINT_URL"] = MLFLOW_SETTINGS.s3_endpoint_url
os.environ["AWS_REGION"] = MLFLOW_SETTINGS.aws_region

## Configuration MLflow

In [6]:
def get_or_restore_experiment(experiment_name: str) -> Experiment:
    client = MlflowClient()

    experiment = client.get_experiment_by_name(experiment_name)

    if experiment is None:
        client.create_experiment(experiment_name)
        return client.get_experiment_by_name(experiment_name)

    if experiment.lifecycle_stage == "deleted":
        client.restore_experiment(experiment.experiment_id)

    return experiment

In [7]:
MLFLOW_EXPERIMENT_NAME = "cqt_mlp"

mlflow.set_tracking_uri(MLFLOW_SETTINGS.tracking_uri)

experiment = get_or_restore_experiment(MLFLOW_EXPERIMENT_NAME)

mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

print("MLflow tracking URI :", mlflow.get_tracking_uri())
print("Experiment ID       :", experiment.experiment_id)
print("Experiment name     :", experiment.name)
print("Lifecycle stage     :", experiment.lifecycle_stage)

MLflow tracking URI : http://localhost:5000
Experiment ID       : 3
Experiment name     : cqt_mlp
Lifecycle stage     : active


## Chargement des données

In [8]:
settings_standard = DatasetBuilderPipelineSettings()
settings_standard.output_dataset_name = "guitar_set_standard"
settings_standard.datasets_used = (GUITAR_SET_SETTINGS.name,)
settings_standard.preprocessing_pipeline_id = None
settings_standard.train_size = 0.7
settings_standard.validation_size = 0.1
settings_standard.test_size = 0.2
settings_standard.random_state = 73
settings_standard.shuffle = True
settings_standard.use_context_window = False
settings_standard.context_size = 11

dataset_builder_pipeline = DatasetBuilderPipeline(
    logging.getLogger(), settings=settings_standard
)

train_dataset, validation_dataset, test_dataset = dataset_builder_pipeline.run()

In [9]:
train_features, train_target = train_dataset
X_train = train_features.values
y_train = train_target.values

validation_features, validation_target = validation_dataset
X_validation = validation_features.values
y_validation = validation_target.values

test_features, test_target = test_dataset
X_test = test_features.values
y_test = test_target.values

feature_names = train_features.columns.to_list()
target_names = train_target.columns.to_list()

print(
    f"Dimension jeu d'entrainement : features={X_train.shape}, target={y_train.shape}"
)
print(
    f"Dimension jeu de validation  : features={X_validation.shape}, target={y_validation.shape}"
)
print(f"Dimension jeu de test        : features={X_test.shape}, target={y_test.shape}")
print()

print("Noms des features :", feature_names)
print("Noms des targets  :", target_names)
print()

Dimension jeu d'entrainement : features=(342529, 84), target=(342529, 49)
Dimension jeu de validation  : features=(38591, 84), target=(38591, 49)
Dimension jeu de test        : features=(91440, 84), target=(91440, 49)

Noms des features : ['cqt_0', 'cqt_1', 'cqt_2', 'cqt_3', 'cqt_4', 'cqt_5', 'cqt_6', 'cqt_7', 'cqt_8', 'cqt_9', 'cqt_10', 'cqt_11', 'cqt_12', 'cqt_13', 'cqt_14', 'cqt_15', 'cqt_16', 'cqt_17', 'cqt_18', 'cqt_19', 'cqt_20', 'cqt_21', 'cqt_22', 'cqt_23', 'cqt_24', 'cqt_25', 'cqt_26', 'cqt_27', 'cqt_28', 'cqt_29', 'cqt_30', 'cqt_31', 'cqt_32', 'cqt_33', 'cqt_34', 'cqt_35', 'cqt_36', 'cqt_37', 'cqt_38', 'cqt_39', 'cqt_40', 'cqt_41', 'cqt_42', 'cqt_43', 'cqt_44', 'cqt_45', 'cqt_46', 'cqt_47', 'cqt_48', 'cqt_49', 'cqt_50', 'cqt_51', 'cqt_52', 'cqt_53', 'cqt_54', 'cqt_55', 'cqt_56', 'cqt_57', 'cqt_58', 'cqt_59', 'cqt_60', 'cqt_61', 'cqt_62', 'cqt_63', 'cqt_64', 'cqt_65', 'cqt_66', 'cqt_67', 'cqt_68', 'cqt_69', 'cqt_70', 'cqt_71', 'cqt_72', 'cqt_73', 'cqt_74', 'cqt_75', 'cqt_76', 

## Normalisation des données

On utilisera un **RobustScaler** du fait que la distribution des données soit fortement censurée (confère le notebook [23_eda_dataset_frame_wise](./23_eda_dataset_frame_wise.ipynb)).

In [10]:
scaler = RobustScaler()

X_train_normalized = scaler.fit_transform(X_train)
X_validation_normalized = scaler.transform(X_validation)
X_test_normalized = scaler.transform(X_test)

## Définition des modèles

### Callbacks

In [11]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

### MLP

Ce modèle repose sur un perceptron multicouche (MLP) constitué de plusieurs couches entièrement connectées. Il reçoit en entrée un vecteur de caractéristiques extrait du signal audio et prédit les notes présentes pour la trame considérée.

Chaque couche cachée est composée de neurones entièrement connectés suivis d'une activation ELU. L'apprentissage est stabilisé par l'utilisation de la Batch Normalization, tandis qu'une régularisation L2 limite le risque de surapprentissage.

La couche de sortie utilise une activation sigmoïde afin de réaliser une classification multi-label, où chaque neurone estime indépendamment la probabilité de présence d'une note.

L'entraînement est effectué avec une fonction de coût Binary Cross-Entropy et les performances sont évaluées à l'aide des métriques Precision, Recall et F1-score micro.

Plusieurs variantes ont été étudiées :

- AdamW : architecture de référence optimisée avec l'algorithme AdamW.
- SGD + Nesterov : même architecture entraînée avec une descente de gradient stochastique utilisant le momentum de Nesterov.
- Squared : architecture composée de trois couches cachées de même taille afin d'étudier l'influence de la largeur du réseau.
- Deep : architecture plus profonde visant à augmenter la capacité de représentation du modèle.

In [12]:
input_dim = X_train.shape[1]
output_dim = y_train.shape[1]
print("Imput dimension  :", input_dim)
print("Output dimension :", output_dim)

Imput dimension  : 84
Output dimension : 49


In [13]:
def build_mlp_model(
    input_dim: int,
    output_dim: int,
    hidden_units: list[int],
    optimizer: tf.keras.optimizers.Optimizer,
    activation_layer: tf.keras.layers.Layer | None = None,
    use_batch_norm: bool = True,
    dropout_rates: list[float] | None = None,
    weight_decay: float = 1e-4,
):
    if activation_layer is None:
        activation_layer = tf.keras.layers.ELU()

    if dropout_rates is None:
        dropout_rates = [0.0] * len(hidden_units)

    if len(dropout_rates) != len(hidden_units):
        raise ValueError("'dropout_rates' and 'hidden_units' must have the same length")

    he_init = tf.keras.initializers.HeNormal()

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(input_dim,)))

    for units, dropout_rate in zip(hidden_units, dropout_rates):
        model.add(
            tf.keras.layers.Dense(
                units,
                kernel_initializer=he_init,
                kernel_regularizer=tf.keras.regularizers.l2(weight_decay),
                use_bias=not use_batch_norm,
            )
        )

        if use_batch_norm:
            model.add(tf.keras.layers.BatchNormalization())

        model.add(activation_layer)

        if dropout_rate > 0:
            model.add(tf.keras.layers.Dropout(dropout_rate))

    model.add(
        tf.keras.layers.Dense(
            output_dim,
            activation="sigmoid",
        )
    )

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.F1Score(
                average="micro",
                threshold=0.5,
                name="f1_micro",
            ),
        ],
    )

    return model


build_mlp_adamw_model = functools.partial(
    build_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    hidden_units=[512, 256, 128],
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    dropout_rates=None,
    weight_decay=1e-4,
)

build_mlp_sgd_nesterov_model = functools.partial(
    build_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    hidden_units=[512, 256, 128],
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.05,
        momentum=0.9,
        nesterov=True,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    dropout_rates=None,
    weight_decay=1e-4,
)

build_mlp_squared_model = functools.partial(
    build_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    hidden_units=[256, 256, 256],
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    dropout_rates=None,
    weight_decay=1e-4,
)

build_mlp_deep_model = functools.partial(
    build_mlp_model,
    input_dim=input_dim,
    output_dim=output_dim,
    hidden_units=[1024, 512, 256, 128],
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=1e-3,
        weight_decay=1e-4,
    ),
    activation_layer=tf.keras.layers.ELU(),
    use_batch_norm=True,
    dropout_rates=None,
    weight_decay=1e-4,
)

## Evaluation des modèles

La transcription audio → MIDI est formulée comme un problème de classification multi-label frame-wise :

- chaque ligne correspond à une frame temporelle ;
- chaque colonne correspond à une note MIDI ;
- plusieurs notes peuvent être actives simultanément.

Exemple :

| Frame | C4 | D4 | E4 | F4 |
|---------|----|----|----|----|
| t₁ | 1 | 0 | 1 | 0 |
| t₂ | 0 | 0 | 1 | 1 |

Une erreur peut donc être commise :
- sur une note spécifique (par exemple une note non détectée),
- sur une frame complète (par exemple une frame non parfaitement transcrite)
- sur la structure musicale globale (par exemple une note transcrite discontinuement qui devrait être continue).

Nous utilisons donc plusieurs métriques complémentaires pour capturer tous ces aspects.

### F1-score Micro

**Définition :** Le F1-score est la moyenne harmonique entre la précision et le rappel.
Dans le cas **micro**, tous les labels de toutes les frames sont regroupés avant calcul.

$$
Precision_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FP}
$$

$$
Recall_{micro}
=
\frac{\sum TP}
{\sum TP + \sum FN}
$$

$$
F1_{micro}
=
2 \cdot
\frac{
Precision_{micro}
\cdot
Recall_{micro}
}
{
Precision_{micro}
+
Recall_{micro}
}
$$

**Utilité :** Cette métrique répond à la question : "Quelle est la qualité globale de la transcription ?"
Toutes les prédictions sont considérées ensemble, c'est à dire, toutes les notes, toutes les frames, tous les morceaux.

**Interprétation :**

| Valeur | Interprétation |
| :- | :- |
| 1.0 | transcription parfaite |
| > 0.9 | excellente |
| 0.8 - 0.9 | très bonne |
| 0.7 - 0.8 | correcte |
| < 0.7 | amélioration nécessaire |

Le F1 micro constitue la métrique principale pour comparer plusieurs modèles.

### F1-score Macro

**Définition :** On calcule d'abord un F1-score pour chaque note MIDI $F1_k$, puis on effectue la moyenne :

$$
F1_{macro}
=
\frac{1}{K}
\sum_{k=1}^{K}
F1_k
$$

où $k$ représente le nombre total de notes MIDI modélisées.

**Utilité :** Le F1 micro est dominé par les notes les plus fréquentes.
Le F1 macro donne le même poids à une note très fréquente et à une note très rare.
Il permet donc d'évaluer la capacité du modèle à généraliser sur l'ensemble du registre de la guitare.

**Interprétation :**
Un écart important entre $F1_{micro} \gg F1_{macro}$ indique généralement que les notes fréquentes sont bien reconnues et que les notes rares sont mal reconnues. Ce peut être le signe d'un déséquilibre de classes.

### Precision

**Définition :**

$$
Precision = \frac{TP}{TP + FP}
$$

où TP signifie True Positives et FP signifie False Positives.

**Utilité :** La précision répond à la question : "Quand le modèle prédit une note, a-t-il raison ?". Une faible précision signifie que le modèle ajoute beaucoup de notes inexistantes.

**Interprétation :**
Une précision faible révèle un grand nombre de notes inexistantes et une transcription surchargée.
Une précision élevée indique qu'il y a peu de fausses notes et que la transcription est propre.

### Recall

**Définition :**

$$
Recall = \frac{TP}{TP + FN}
$$

où TP signifie True Positives et FN signifie False Negatives

**Utilité :** Le rappel répond à la question : "Combien de vraies notes le modèle retrouve-t-il ?"

**Interprétation :**
Un recall faible révèle que le modèle oublie des notes et que transcription incomplète.
Un recall élevé montre que davantage de notes sont détectées, parfois au prix de faux positifs supplémentaires.

### Hamming Loss

**Définition :**

$$
HammingLoss = \frac{FP + FN}{N \times K}
$$

avec $N$ le nombre de frames et $K$ le nombre de notes MIDI.

**Utilité :** Cette métrique mesure le taux d'erreur moyen par note et par frame.
Contrairement au F1-score, elle pénalise directement chaque erreur élémentaire.


**Interprétation :**

| Valeur | Signification |
| :- | :- |
| 0 | aucune erreur |
| 0.01 | 1 % d'erreurs |
| 0.05 | 5 % d'erreurs |
| 0.10 | 10 % d'erreurs |

Plus la valeur est faible, meilleur est le modèle.

### Subset Accuracy

**Définition :** Une frame est correcte uniquement si toutes les notes sont correctement prédites.

$$
SubsetAccuracy = \frac{\#\;frames\;parfaites}{\#\;frames}
$$

**Utilité :** Cette métrique est extrêmement stricte.
Elle répond à la question : "Combien de frames sont parfaitement transcrites ?"

**Interprétation :**
Même un très bon modèle obtient souvent une valeur relativement faible.
Cette métrique permet de mesurer la qualité des accords complets.

In [14]:
def compute_ml_metrics(y_true, y_pred):
    return {
        "f1_micro": f1_score(y_true, y_pred, average="micro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_micro": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "precision_macro": precision_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "recall_micro": recall_score(y_true, y_pred, average="micro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "hamming_loss": hamming_loss(y_true, y_pred),
        "subset_accuracy": accuracy_score(y_true, y_pred),
    }

### F1-score par pitch MIDI

**Définition :** Pour chaque note MIDI :

$$
F1_k = 2 \cdot \frac{Precision_k \cdot Recall_k}{Precision_k + Recall_k}
$$

**Utilité :** Le score global peut masquer des difficultés spécifiques.
Certaines notes peuvent être très bien reconnues et d'autres très mal reconnues.

Le F1 par pitch permet d'identifier les zones du manche difficiles, les fréquences mal représentées ou les erreurs de feature engineering.

**Interprétation :** Un graphique F1 par pitch permet de visualiser les notes problématiques, les tendances graves / aigus et les limites du modèle.

In [15]:
def compute_f1_per_pitch(
    y_true,
    y_pred,
    pitch_offset=40,  # /!\ Regarder les settings de la pipeline de prétraitement
):
    scores = []

    for k in range(y_true.shape[1]):
        scores.append(
            {
                "pitch_midi": k + pitch_offset,
                "f1_score": f1_score(
                    y_true[:, k],
                    y_pred[:, k],
                    average="binary",
                    zero_division=0,
                ),
            }
        )

    return pd.DataFrame(scores)

### Pitch Tolerance Accuracy

**Définition :** Une prédiction est considérée correcte si elle est proche de la vraie note :

$$
|Pitch_{pred} - Pitch_{true}| \leq t
$$

où $t$ représente le nombre de demi-tons.

**Utilité :** Une erreur d'un demi-ton est moins grave musicalement qu'une erreur d'une octave.
Le F1-score classique considère pourtant ces deux erreurs comme identiques.
Cette métrique introduit une notion de proximité musicale.

**Interprétation :** Une pitch tolerance élevée indique que le modèle comprend globalement les hauteurs de notes.
Une pitch tolerance faible indique que le modèle commet des erreurs importantes sur les hauteurs de notes.

In [16]:
def pitch_tolerance_accuracy(y_true, y_pred, tolerance=1):
    true_idx = np.where(y_true == 1)
    pred_idx = np.where(y_pred == 1)

    if len(true_idx[0]) == 0:
        return 0.0

    correct = 0

    for i in range(len(true_idx[0])):
        t_frame = true_idx[0][i]
        t_pitch = true_idx[1][i]

        frame_preds = pred_idx[1][pred_idx[0] == t_frame]

        if len(frame_preds) == 0:
            continue

        if np.any(np.abs(frame_preds - t_pitch) <= tolerance):
            correct += 1

    return correct / len(true_idx[0])

### Activation Ratio

**Définition :**
$$
ActivationRatio = \frac{\text{taux d'activation prédit}}{\text{taux d'activation réel}}
$$

**Utilité :** Cette métrique mesure le biais global du modèle.

**Interprétation :**
- $Ratio \approx 1$ : Le modèle produit globalement le bon nombre de notes.
- $Ratio > 1$ : Le modèle sur-prédit, il ajoute trop de notes.
- $Ratio < 1$ : Le modèle sous-prédit, il manque des notes.

In [17]:
def activation_ratio(y_true, y_pred):
    return {
        "true_activation": y_true.mean(),
        "pred_activation": y_pred.mean(),
        "ratio": (y_pred.mean() / (y_true.mean() + 1e-8)),
    }

### Temporal Jitter

**Définition :** Le jitter mesure les variations de prédictions entre frames successives.
Une approximation simple est :

$$
Jitter = mean \left(|y_t - y_{t-1}| \right)
$$

**Utilité :** La transcription frame-wise produit souvent un phénomène appelé *flickering*.
Une note apparaît puis disparaît très rapidement alors qu'elle devrait rester stable.

**Interprétation :**
Un jitter faible indique une transcription stable avec des notes continues.
Un jitter élevé révèle une instabilité temporelle.

In [18]:
def temporal_jitter(y_pred):
    return np.mean(np.abs(np.diff(y_pred, axis=0)))

### Pitch Class Confusion Matrix

**Définition :**
Une note MIDI peut être ramenée à sa classe de hauteur (Pitch Class) : $PitchClass = MIDI \bmod 12$
Les notes séparées d'une ou plusieurs octaves appartiennent donc à la même classe.

**Utilité :** La Pitch Class Confusion Matrix regroupe les notes par nom musical (C, C#, D, D#, E, F, F#, G, G#, A, A#, B).
Elle permet de mettre en évidence des erreurs harmoniques ou tonales.
Deux erreurs peuvent avoir le même impact sur le F1-score, par exemple, prédire E au lieu de F et prédire E au lieu de A#.
Pourtant musicalement, ces erreurs sont très différentes.
La Pitch Class Confusion Matrix permet d'analyser la nature musicale des erreurs plutôt que leur simple quantité.

**Interprétation :**
Une diagonale dominante indique que les classes de hauteur sont correctement reconnues.
Des valeurs importantes hors diagonale indiquent des confusions entre notes voisines, des difficultés dans certaines régions fréquentielles et d'éventuels problèmes liés aux harmoniques de la guitare.

In [19]:
def pitch_class_confusion(y_true, y_pred):
    true_pc = np.where(y_true == 1)[1] % 12
    pred_pc = np.where(y_pred == 1)[1] % 12

    cm = np.zeros((12, 12))

    for t, p in zip(true_pc, pred_pc):
        cm[t, p] += 1

    return cm

In [20]:
def evaluate(
    model,
    X_test,
    y_test,
    label_names,
    pitch_offset=40,  # /!\ Regarder les settings de la pipeline de prétraitement
    threshold=0.5,
):
    y_score = model.predict(X_test, verbose=0)

    y_pred = (y_score >= threshold).astype(np.int32)

    report = classification_report(
        y_test,
        y_pred,
        target_names=label_names,
        output_dict=True,
        zero_division=0,
    )

    metrics = {}
    metrics.update(compute_ml_metrics(y_test, y_pred))

    metrics["pitch_acc_tol_1"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=1)
    metrics["pitch_acc_tol_2"] = pitch_tolerance_accuracy(y_test, y_pred, tolerance=2)

    metrics.update(activation_ratio(y_test, y_pred))

    metrics["jitter"] = temporal_jitter(y_pred)

    df_f1_per_pitch = compute_f1_per_pitch(y_test, y_pred, pitch_offset)

    cm = confusion_matrix(y_test.flatten(), y_pred.flatten())

    pitch_class_cm = pitch_class_confusion(y_test, y_pred)

    artifacts = {
        "y_pred": y_pred,
        "y_score": y_score,
        "confusion_matrix": cm,
        "classification_report": report,
        "f1_per_pitch": df_f1_per_pitch,
        "pitch_class_confusion_matrix": pitch_class_cm,
    }

    return metrics, artifacts

In [21]:
def log_confusion_matrix(cm, artifact_file="confusion_matrix.png"):
    cm_percent = cm / cm.sum().sum() * 100

    plt.figure(figsize=(7, 5))

    sns.heatmap(
        cm_percent,
        annot=True,
        fmt=".2f",
        cmap="cividis",
        square=True,
        linewidths=0.6,
        linecolor="white",
        annot_kws={"size": 10},
    )

    plt.title("Matrice de confusion")
    plt.xlabel("Predict label")
    plt.ylabel("True label")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [22]:
def log_f1_per_pitch(df_scores, artifact_file="f1_per_pitch.png"):
    plt.figure(figsize=(10, 4))

    plt.plot(
        df_scores["pitch_midi"],
        df_scores["f1_score"],
    )

    plt.title("F1-score per Pitch")
    plt.xlabel("MIDI Pitch")
    plt.ylabel("F1-score")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [23]:
def log_precision_recall_curve(
    y_true, y_score, artifact_file="precision_recall_curve.png"
):
    precision, recall, _ = precision_recall_curve(
        y_true.flatten(),
        y_score.flatten(),
    )

    plt.figure(figsize=(6, 6))

    plt.plot(recall, precision)

    plt.title("Global Precision-Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

### Training History

**Principe :**
À chaque époque d'entraînement, on mesure les performances du modèle sur le jeu d'entraînement et sur le jeu de validation. Les courbes retracent l'évolution des métriques (loss, précision, rappel, F1, etc.) au fil des époques.

**Utilité :**
L'historique d'entraînement permet de répondre à plusieurs questions :

* le modèle continue-t-il à apprendre ?
* l'entraînement a-t-il convergé ?
* le modèle commence-t-il à sur-apprendre ?
* les callbacks (Early Stopping, ReduceLROnPlateau) interviennent-ils au bon moment ?
* davantage d'époques seraient-elles bénéfiques ?

**Interprétation :**

* Loss entraînement diminue et loss validation diminue : apprentissage sain.
* Loss entraînement diminue mais loss validation augmente : sur-apprentissage.
* Loss entraînement et validation stagnent à des valeurs élevées : sous-apprentissage.
* Les métriques de validation continuent à s'améliorer en fin d'entraînement : davantage d'époques pourraient améliorer les performances.
* Les métriques de validation se stabilisent tandis que les métriques d'entraînement continuent à progresser : le modèle atteint probablement sa capacité de généralisation maximale.
* Une forte instabilité des métriques entre époques peut révéler un taux d'apprentissage trop élevé, un batch size trop faible ou un jeu de données insuffisant.

In [24]:
def log_training_history(
    history,
    metric="f1_micro",
    loss="binary_crossentropy",
    artifact_file="training_history.png",
):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Loss
    ax1.plot(history.history["loss"], label="Train loss")
    ax1.plot(history.history["val_loss"], label="Validation loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel(f"Loss ({loss.upper()})")
    ax1.set_title(f"Loss evolution ({loss.upper()})")
    ax1.legend()

    # Metric
    ax2.plot(history.history[metric], label=f"Train {metric.upper()}")
    ax2.plot(history.history[f"val_{metric}"], label=f"Validation {metric.upper()}")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel(f"{metric.upper()}")
    ax2.set_title(f"{metric.upper()} evolution")
    ax2.legend()

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

## Expériences

In [25]:
def optimize_threshold(
    model,
    X_validation,
    y_validation,
    thresholds=np.arange(0.05, 0.951, 0.01),
):
    y_score = model.predict(X_validation, verbose=0)

    results = []

    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(np.uint8)

        results.append(
            {
                "threshold": threshold,
                "f1_micro": f1_score(
                    y_validation,
                    y_pred,
                    average="micro",
                    zero_division=0,
                ),
                "f1_macro": f1_score(
                    y_validation,
                    y_pred,
                    average="macro",
                    zero_division=0,
                ),
            }
        )

    results = pd.DataFrame(results)

    best_idx = results["f1_micro"].idxmax()

    return (
        float(results.loc[best_idx, "threshold"]),
        float(results.loc[best_idx, "f1_micro"]),
        results,
    )

In [26]:
def log_threshold_search(results: pd.DataFrame, artifact_file="threshold_search.png"):
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(
        results["threshold"],
        results["f1_micro"],
        marker="x",
    )

    ax.set_xlabel("Threshold")
    ax.set_ylabel("F1 score")
    ax.set_title("Threshold optimization")

    plt.tight_layout()
    mlflow.log_figure(plt.gcf(), artifact_file)
    plt.close()

In [27]:
def run_experiment(model_factory, run_name, tags):

    print("=" * 80)
    print(f"[{datetime.now()}] Starting run: {run_name}")

    with mlflow.start_run(run_name=run_name) as run:
        run_id = run.info.run_id
        print(f"[{datetime.now()}] MLflow run_id: {run_id}")

        mlflow.set_tags(tags)

        print(f"[{datetime.now()}] Building model...")
        model = model_factory()

        print(f"[{datetime.now()}] Logging dataset name and id...")
        mlflow.log_params(
            {"dataset_id": dataset_builder_pipeline.pipeline_metadata["_id"]}
        )
        mlflow.log_params({"dataset_name": settings_standard.output_dataset_name})

        print(f"[{datetime.now()}] Logging model config...")
        config_path = ARTIFACT_DIR / "model_config.json"
        with open(config_path, "w") as f:
            json.dump(model.get_config(), f)
        mlflow.log_artifact(str(config_path))

        print(f"[{datetime.now()}] Training model...")
        t0 = perf_counter()
        history = model.fit(
            X_train_normalized,
            y_train,
            validation_data=(X_validation_normalized, y_validation),
            batch_size=32,
            epochs=200,
            callbacks=[early_stopping, reduce_lr],
            verbose=1,
        )
        fitting_time = perf_counter() - t0
        mlflow.log_metric("fitting_time", fitting_time)
        print(f"[{datetime.now()}] Training completed ({fitting_time:.1f}s)")

        print(f"[{datetime.now()}] Logging best validation score...")
        best_epoch = np.argmax(history.history["val_f1_micro"])
        mlflow.log_metric(
            "best_validation_f1_micro", history.history["val_f1_micro"][best_epoch]
        )
        mlflow.log_metric(
            "best_validation_precision", history.history["val_precision"][best_epoch]
        )
        mlflow.log_metric(
            "best_validation_recall", history.history["val_recall"][best_epoch]
        )

        print(f"[{datetime.now()}] Searching optimal threshold...")
        best_threshold, validation_score, threshold_results = optimize_threshold(
            model,
            X_validation_normalized,
            y_validation,
        )
        print(
            f"[{datetime.now()}] Best threshold : {best_threshold} (validation_f1_micro={validation_score:.4f})"
        )

        mlflow.log_param("prediction_threshold", best_threshold)
        mlflow.log_metric("validation_f1_micro_best_threshold", validation_score)

        print(f"[{datetime.now()}] Saving threshold search...")
        log_threshold_search(threshold_results)
        threshold_csv_path = ARTIFACT_DIR / "threshold_search.csv"
        threshold_results.to_csv(threshold_csv_path, index=False)
        mlflow.log_artifact(str(threshold_csv_path))

        print(f"[{datetime.now()}] Evaluating on test set...")
        metrics, artifacts = evaluate(
            model=model,
            X_test=X_test_normalized,
            y_test=y_test,
            label_names=target_names,
            threshold=best_threshold,
        )
        print(f"[{datetime.now()}] Evaluation completed ({len(metrics)} metrics)")

        print(f"[{datetime.now()}] Logging metrics...")
        mlflow.log_metrics(metrics)

        print(f"[{datetime.now()}] Saving classification report...")
        report_path = ARTIFACT_DIR / "classification_report.json"
        with open(report_path, "w") as f:
            json.dump(
                artifacts["classification_report"],
                f,
                indent=2,
            )
        mlflow.log_artifact(str(report_path))

        print(f"[{datetime.now()}] Saving pitch metrics...")
        f1_pitch_csv_path = ARTIFACT_DIR / "f1_per_pitch.csv"
        artifacts["f1_per_pitch"].to_csv(
            f1_pitch_csv_path,
            index=False,
        )
        mlflow.log_artifact(str(f1_pitch_csv_path))

        print(f"[{datetime.now()}] Logging confusion matrix...")
        log_confusion_matrix(artifacts["confusion_matrix"])

        print(f"[{datetime.now()}] Logging F1-per-pitch plot...")
        log_f1_per_pitch(artifacts["f1_per_pitch"])

        print(f"[{datetime.now()}] Logging precision-recall curve...")
        log_precision_recall_curve(y_test, artifacts["y_score"])

        print(f"[{datetime.now()}] Logging training history...")
        log_training_history(history)

        print(f"[{datetime.now()}] Logging scaler...")
        mlflow.sklearn.log_model(scaler, name="scaler")
        print(f"[{datetime.now()}] Scaler logged successfully")

        print(f"[{datetime.now()}] Logging tensorflow model...")
        input_example = X_train_normalized[:5]
        prediction = model.predict(input_example)
        signature = infer_signature(input_example, prediction)
        mlflow.tensorflow.log_model(
            model=model,
            name="model",
            signature=signature,
            input_example=input_example,
        )
        print(f"[{datetime.now()}] Model logged successfully")

        print("=" * 80)
        print("Run completed")
        print(f"Run ID : {run_id}")

        print("\nMain metrics:")

        summary_metrics = [
            "test_f1_micro",
            "test_f1_macro",
            "test_precision_micro",
            "test_recall_micro",
        ]

        for metric in summary_metrics:
            if metric in metrics:
                print(f"{metric}: {metrics[metric]:.4f}")

        print("=" * 80)

In [28]:
experiments = [
    (
        build_mlp_adamw_model,
        "mlp_adamw_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "tensorflow",
            "model": "mlp_adamw",
        },
    ),
    (
        build_mlp_sgd_nesterov_model,
        "mlp_sgd_nesterov_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "tensorflow",
            "model": "mlp_nesterov",
        },
    ),
    (
        build_mlp_squared_model,
        "mlp_squared_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "tensorflow",
            "model": "mlp_squared",
        },
    ),
    (
        build_mlp_deep_model,
        "mlp_deep_cqt",
        {
            "task": "audio_to_midi",
            "representation": "cqt",
            "model_family": "tensorflow",
            "model": "mlp_deep",
        },
    ),
]

for model_factory, run_name, tags in experiments:
    run_experiment(model_factory, run_name, tags)

[2026-07-28 01:40:05.749518] Starting run: mlp_adamw_cqt
[2026-07-28 01:40:06.523147] MLflow run_id: 3916901bb35a41159bb671b42bae00d2
[2026-07-28 01:40:06.582069] Building model...
[2026-07-28 01:40:06.646720] Logging dataset name and id...
[2026-07-28 01:40:06.785110] Logging model config...
[2026-07-28 01:40:08.344087] Training model...
Epoch 1/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - f1_micro: 0.7268 - loss: 0.0724 - precision: 0.8291 - recall: 0.6470 - val_f1_micro: 0.7680 - val_loss: 0.0468 - val_precision: 0.8454 - val_recall: 0.7036 - learning_rate: 0.0010
Epoch 2/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 22s 2ms/step - f1_micro: 0.7792 - loss: 0.0533 - precision: 0.8635 - recall: 0.7100 - val_f1_micro: 0.7901 - val_loss: 0.0419 - val_precision: 0.8691 - val_recall: 0.7243 - learning_rate: 0.0010
Epoch 3/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 69s 5ms/step - f1_micro: 0.7878 - loss: 0.0511 - precision: 0.8682 - recall: 0.7211 - val_f1_micro: 0.7964 - val_loss: 0.0405 - val_pr

2026/07/28 02:55:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/28 02:55:04 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/07/28 02:55:10 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-07-28 02:55:10.454884] Scaler logged successfully
[2026-07-28 02:55:10.455169] Logging tensorflow model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


2026/07/28 02:55:16 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
[2026-07-28 02:55:17.395207] Model logged successfully
Run completed
Run ID : 3916901bb35a41159bb671b42bae00d2

Main metrics:
🏃 View run mlp_adamw_cqt at: http://localhost:5000/#/experiments/3/runs/3916901bb35a41159bb671b42bae00d2
🧪 View experiment at: http://localhost:5000/#/experiments/3
[2026-07-28 02:55:17.469230] Starting run: mlp_sgd_nesterov_cqt
[2026-07-28 02:55:17.528044] MLflow run_id: e53ce3deab2f4b8d8245c62cd85f992f
[2026-07-28 02:55:17.584493] Building model...
[2026-07-28 02:55:17.624727] Logging dataset name and id...
[2026-07-28 02:55:17.744477] Logging model config...
[2026-07-28 02:55:17.828236] Training model...
Epoch 1/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 25s 2ms/step - f1_micro: 0.7076 - loss: 0.1355 - precision: 0.8296 - recall: 0.6169 - val_f1_micro: 0.7544 - val_loss: 0.0654 - val_precision: 0.8458 - val_recall: 0.6809 - learning_rate: 0.0500
Epoch 2/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 24s 2ms/step - f1_micro: 0.7717 - loss

...

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
[2026-07-28 03:04:57.002119] Model logged successfully
Run completed
Run ID : 58a8ee0fc70040f780599d65fcf8104d

Main metrics:
🏃 View run mlp_squared_cqt at: http://localhost:5000/#/experiments/3/runs/58a8ee0fc70040f780599d65fcf8104d
🧪 View experiment at: http://localhost:5000/#/experiments/3
[2026-07-28 03:04:57.073975] Starting run: mlp_deep_cqt
[2026-07-28 03:04:57.133943] MLflow run_id: cd2545b5c0bd4e9fa35f828df9c0d94a
[2026-07-28 03:04:57.191309] Building model...
[2026-07-28 03:04:57.251904] Logging dataset name and id...
[2026-07-28 03:04:57.366612] Logging model config...
[2026-07-28 03:04:57.418392] Training model...
Epoch 1/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 45s 4ms/step - f1_micro: 0.7091 - loss: 0.0795 - precision: 0.8175 - recall: 0.6261 - val_f1_micro: 0.7556 - val_loss: 0.0507 - val_precision: 0.8142 - val_recall: 0.7049 - learning_rate: 0.0010
Epoch 2/200
10705/10705 ━━━━━━━━━━━━━━━━━━━━ 43s 4ms/step - f1_micro: 0.7700 - loss: 0.05

2026/07/28 03:13:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/28 03:13:19 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2026/07/28 03:13:22 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[2026-07-28 03:13:22.580377] Scaler logged successfully
[2026-07-28 03:13:22.580600] Logging tensorflow model...
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


2026/07/28 03:13:28 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
[2026-07-28 03:13:29.130821] Model logged successfully
Run completed
Run ID : cd2545b5c0bd4e9fa35f828df9c0d94a

Main metrics:
🏃 View run mlp_deep_cqt at: http://localhost:5000/#/experiments/3/runs/cd2545b5c0bd4e9fa35f828df9c0d94a
🧪 View experiment at: http://localhost:5000/#/experiments/3


### Analyse des résultats

#### Comparaison des modèles

Les graphiques suivants présentent les résultats pour l'ensemble des modèles entraînés.

| F1_micro mean | Best validation F1_micro mean |
| :-: | :-: |
| ![f1_micro_mean](../output/cqt_mlp/mlp_results/f1_micro.png) | ![best_validation_f1_micro_mean](../output/cqt_mlp/mlp_results/best_validation_f1_micro.png) |

On observe que le modèle mlp_adamw_cqt offre les meilleurs résultats. La transcription est bonne (0.8 < f1_micro < 0.9). Les variantes du modèle MLP avec optimseur AdamW ne semble pas améliorer le modèle.

| F1_macro mean | F1_micro per pitch |
| :-: | :-: |
| ![f1_macro_mean](../output/cqt_mlp/mlp_results/f1_macro.png) | ![f1_micro_per_pitch](../output/cqt_mlp/mlp_results/f1_per_pitch.png) |

Le F1_macro est inférieur au F1_micro (0.63 < 0.81), pouvant indiquer que les notes rares sont moins bien reconnues. Le graphiques du F1_micro par pitch montre que les notes ayant un pitch de 73 ou 75 sont moins bien reconnues.

| Activation ratio | Pitch accuracy tolérance 1 demi-ton | Pitch accuracy tolérance 2 demi-ton |
| :-: | :-: | :-: |
| ![activation_ratio](../output/cqt_mlp/mlp_results/ratio.png) | ![pitch_accuracy_tolerance_1](../output/cqt_mlp/mlp_results/pitch_acc_tol_1.png) | ![pitch_accuracy_tolerance_2](../output/cqt_mlp/mlp_results/pitch_acc_tol_2.png) |

Le modèle mlp_adamw_cqt a le ratio d'activation le plus proche de 1. Ce ratio est légérement inférieur à 1 ce qui montre sous-prédit, il manque des notes.
Les métriques de pitch tolérance sont également en faveur du modèle mlp_adamw_cqt. Ce modèle est plus précis sur la hauteur de la note lors d'une activation.

| Subset accuracy | Hamming loss | Jitter |
| :-: | :-: | :-: |
| ![subset_accuracy](../output/cqt_mlp/mlp_results/subset_accuracy.png) | ![hamming_loss](../output/cqt_mlp/mlp_results/hamming_loss.png) | ![jitter](../output/cqt_mlp/mlp_results/jitter.png) |

Le modèle mlp_adamw_cqt obtient le meilleur subset_accuracy, environ 55% des frames sont parfaitement détectées.
Il a également le meilleur hamming_loss, environ 2% d'erreurs.
Le modèle mlp_adamw_cqt a un très bon jitter, montrant la stabilté de la transcription.

> Le modèle mlp_adamw_cqt est le meilleur modèle, il montre de plus de meilleurs performance que le modèle HistGradientBoosting.

#### Analyse du meilleur modèle

| Confusion matrix | Learning curve | Precision recall curve |
| :-: | :-: | :-: |
| ![confusion_matrix](../output/cqt_mlp/mlp_results/confusion_matrix.png) | ![learning_curve](../output/cqt_mlp/mlp_results/training_history.png) | ![precision_recall_curve](../output/cqt_mlp/mlp_results/precision_recall_curve.png) |

La matrice de confusion montre que le modèle génère moins de 1% de faux négatifs, il détecte mieux les activations que le modèle HistGradientBoosting.

L'historique d'entrainement montre que les métriques loss et F1-micro convergent rapidement, sans instabilité ni divergence sur les ensembles d'entraînement et de validation. On observe que la loss de validation reste constamment inférieure à la loss d'entraînement, ce comportement peut être dû à l'utilisation de la normalisation l2. On ne constate ausune remontée de la loss de validation en fin d'entraînement, ce qui indique l'absence de surapprentissage. L'écart entre les deux courbes est d'environ 1,5 point de F1. Ce qui montre une bonne capacité de généralisation. Enfin on observe que les courbes atteingnent un plateau aux alentours de 80 époques, pourtant l'entraînement a duré 200 époques. Le Early Stoping ne s'est pas déclanché, il faudra ajouter le paramètre `min_delta=1e-4` pour les prochianes expérimentations.

La courbe precision_recall montre que le modèle est meilleur que le hazard, il détecte correctement les activations.

> Globalement le modèle est bon pour une transcription audio vers midi, il est meilleur que HistGradientBoosting pour toutes les mesures. Nous continuerons notre exploration en ajoutant des couches de convolution.